In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
metrics = pd.read_csv("../data/processed/model_metrics.csv")
predictions = pd.read_csv("../data/processed/model_predictions.csv")

predictions["Date"] = pd.to_datetime(predictions["Date"])

print("MODEL METRICS")
display(metrics)

print("\nPREDICTIONS SHAPE:", predictions.shape)
display(predictions.head())

MODEL METRICS


,MAE,RMSE,R2,Model,Experiment,Feature_Count,CV_RMSE
0,4.338222,5.450728,-1.303246,Persistence,baseline,1,NaN
1,3.094638,3.810527,-0.125645,RandomForest,conventional_only,48,7.909614
2,3.094638,3.810527,-0.125645,RandomForest,conventional_plus_behavioral,48,7.909614



PREDICTIONS SHAPE: (255, 5)


,Date,Actual_ESS_target,Persistence,Conventional,Conventional_Behavioral
0,2005-03-01,43.227530,54.956931,51.994985,51.994985
1,2005-04-01,46.076329,54.603297,51.888134,51.888134
2,2005-05-01,49.006414,55.300261,51.649821,51.649821
3,2005-06-01,58.940236,43.227530,51.734123,51.734123
4,2005-07-01,59.772947,46.076329,51.989740,51.989740


In [3]:
predictions.describe()

,Date,Actual_ESS_target,Persistence,Conventional,Conventional_Behavioral
count,255,255.000000,255.000000,255.000000,255.000000
mean,2015-10-01 08:39:31.764705792,50.212267,50.292275,49.788241,49.788241
min,2005-03-01 00:00:00,40.775966,40.775966,45.095571,45.095571
25%,2010-06-16 00:00:00,47.803573,47.819454,48.500215,48.500215
50%,2015-10-01 00:00:00,50.056774,50.251098,50.485872,50.485872
75%,2021-01-16 12:00:00,52.805666,52.850749,50.997236,50.997236
max,2026-05-01 00:00:00,59.772947,59.772947,52.046988,52.046988
std,NaN,3.598633,3.627029,1.647986,1.647986


In [4]:
actual = predictions["Actual_ESS_target"]

results = []

models = {
    "Persistence Baseline": "Persistence",
    "Conventional": "Conventional",
    "Conventional + Behavioral": "Conventional_Behavioral"
}

for model_name, column in models.items():

    predicted = predictions[column]

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = mean_squared_error(
        actual,
        predicted
    ) ** 0.5

    r2 = r2_score(
        actual,
        predicted
    )

    results.append({
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4)
    })

results_df = pd.DataFrame(results)

results_df

,Model,MAE,RMSE,R2
0,Persistence Baseline,4.3382,5.4507,-1.3032
1,Conventional,3.0946,3.8105,-0.1256
2,Conventional + Behavioral,3.0946,3.8105,-0.1256


In [5]:
best_model = results_df.loc[
    results_df["RMSE"].idxmin()
]

print("BEST MODEL")
print(best_model)

BEST MODEL
Model    Conventional
MAE            3.0946
RMSE           3.8105
R2            -0.1256
Name: 1, dtype: object


In [6]:
conventional_rmse = results_df.loc[
    results_df["Model"] == "Conventional",
    "RMSE"
].iloc[0]

behavioral_rmse = results_df.loc[
    results_df["Model"] == "Conventional + Behavioral",
    "RMSE"
].iloc[0]

improvement = (
    (conventional_rmse - behavioral_rmse)
    / conventional_rmse
) * 100

print(f"Conventional RMSE: {conventional_rmse:.4f}")
print(f"Conventional + Behavioral RMSE: {behavioral_rmse:.4f}")
print(f"Behavioral improvement: {improvement:.2f}%")

Conventional RMSE: 3.8105
Conventional + Behavioral RMSE: 3.8105
Behavioral improvement: 0.00%


In [7]:
print("MIRAI MODEL ANALYSIS SUMMARY")
print("-" * 40)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']}: "
        f"MAE={row['MAE']}, "
        f"RMSE={row['RMSE']}, "
        f"R²={row['R2']}"
    )

MIRAI MODEL ANALYSIS SUMMARY
----------------------------------------
Persistence Baseline: MAE=4.3382, RMSE=5.4507, R²=-1.3032
Conventional: MAE=3.0946, RMSE=3.8105, R²=-0.1256
Conventional + Behavioral: MAE=3.0946, RMSE=3.8105, R²=-0.1256
